# ACTIVIDAD II - STUDENT DROPOUT CLASSIFICATION
**Universidad de la Costa | Data Mining**  
**Docente:** Jose Escorcia-Gutierrez, Ph.D.

---
**Integrantes:**
- Carlos Andres Barreto Castilla

In [ ]:
# CELDA 1 - Instalacion e importaciones
# Descomenta si usas entorno local:
!pip install pandas numpy scikit-learn matplotlib seaborn

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay,
    roc_auc_score, roc_curve, classification_report
)

import warnings
warnings.filterwarnings('ignore')
np.random.seed(42)
print("Librerias importadas correctamente")


ModuleNotFoundError: No module named 'pandas'

## Carga de Datos
- **Dataset sintetico** (Actividad I) → 70% calificacion
- **Dataset real** (adjunto por el profesor) → 30% calificacion

In [ ]:
# CELDA 2 - Generacion del dataset sintetico (Actividad I)
def generar_dataset_sintetico(n=600, seed=42):
    np.random.seed(seed)
    edad      = np.random.randint(16, 30, n).astype(float)
    genero    = np.random.choice(['Masculino','Femenino','Otro'], n, p=[.48,.48,.04])
    ciudad    = np.random.choice(['Barranquilla','Bogota','Medellin','Cartagena','Otra'], n, p=[.40,.20,.15,.15,.10])
    prom_bach = np.random.normal(3.4, 0.6, n).clip(0, 5)
    punt_adm  = np.random.normal(55, 12, n).clip(10, 100)
    prom_sem1 = np.random.normal(3.2, 0.7, n).clip(0, 5)
    nivel     = np.random.choice(['Estrato 1','Estrato 2','Estrato 3','Estrato 4','Estrato 5'], n, p=[.20,.30,.25,.15,.10])
    beca      = np.random.choice(['Si','No'], n, p=[.35,.65])
    prestamo  = np.random.choice(['Si','No'], n, p=[.40,.60])
    apoyo     = np.random.choice(['Ninguno','Parcial','Completo'], n, p=[.45,.35,.20])

    riesgo = (
        (prom_sem1 < 3.0).astype(int) * 2 +
        (prom_bach < 3.0).astype(int) +
        (nivel == 'Estrato 1').astype(int) +
        (nivel == 'Estrato 2').astype(int) +
        (beca == 'No').astype(int) +
        (apoyo == 'Ninguno').astype(int) +
        (punt_adm < 45).astype(int)
    )
    prob      = 1 / (1 + np.exp(-(riesgo - 3)))
    desercion = np.where(np.random.rand(n) < prob, 'Si', 'No')

    df = pd.DataFrame({
        'edad': edad, 'genero': genero, 'ciudad_origen': ciudad,
        'promedio_bachillerato': prom_bach.round(2),
        'puntaje_admision': punt_adm.round(1),
        'promedio_primer_sem': prom_sem1.round(2),
        'nivel_socioeconomico': nivel,
        'beca': beca, 'prestamo': prestamo,
        'apoyo_financiero': apoyo, 'desercion': desercion
    })
    for col in ['edad','promedio_bachillerato','puntaje_admision','promedio_primer_sem']:
        df.loc[np.random.choice(df.index, int(n*.08), replace=False), col] = np.nan
    nb = int(n*.015)
    df.loc[np.random.choice(df.index, nb, replace=False), 'promedio_primer_sem'] = np.random.uniform(0, .5, nb)
    df.loc[np.random.choice(df.index, nb, replace=False), 'puntaje_admision']    = np.random.uniform(95, 120, nb)
    return df

df_syn = generar_dataset_sintetico()
print(f"Dataset sintetico: {df_syn.shape}")
print(df_syn['desercion'].value_counts())
df_syn.head()


In [ ]:
# CELDA 3 - Carga del dataset real (adjunto por el profesor)
# Si estas en Colab, sube el archivo primero:
# from google.colab import files; files.upload()

try:
    df_real = pd.read_csv('student_dropout__1_.csv')
    df_real = df_real.rename(columns={'Target': 'desercion'})
    df_real = df_real[df_real['desercion'].isin(['Dropout','Graduate'])].copy()
    df_real['desercion'] = df_real['desercion'].map({'Graduate': 'No', 'Dropout': 'Si'})
    print(f"Dataset real cargado: {df_real.shape}")
    print(df_real['desercion'].value_counts())
    display(df_real.head())
except FileNotFoundError:
    print("Archivo no encontrado. Sube 'student_dropout__1_.csv' al entorno.")
    df_real = None


## Funcion Principal
Encapsula los **10 pasos** de la actividad. Se ejecuta sobre cualquier dataset.

In [ ]:
# CELDA 4 - Funcion principal: 10 pasos de la actividad
def run_pipeline(df_input, dataset_name="Sintetico"):
    print(f"\n{'='*60}")
    print(f"  DATASET: {dataset_name}")
    print(f"{'='*60}")

    df = df_input.copy()
    target_col = 'desercion'

    # -------------------------------------------------------
    # PASO 1: Limpieza de datos
    # -------------------------------------------------------
    print("\n[PASO 1] LIMPIEZA DE DATOS")
    print(f"  Shape inicial : {df.shape}")
    print(f"  Duplicados    : {df.duplicated().sum()}")
    df.drop_duplicates(inplace=True)
    print(f"  Shape final   : {df.shape}")
    print(f"  Nulos totales : {df.isnull().sum().sum()}")
    print("  Distribucion objetivo:")
    print(df[target_col].value_counts())

    # -------------------------------------------------------
    # PASO 2: EDA
    # -------------------------------------------------------
    print("\n[PASO 2] EDA - ANALISIS EXPLORATORIO")
    num_cols = df.select_dtypes(include='number').columns.tolist()
    cat_cols = df.select_dtypes(include='object').columns.difference([target_col]).tolist()

    print("\nEstadisticas descriptivas (numericas):")
    display(df[num_cols].describe().round(2))

    print("\nTablas de frecuencia (categoricas):")
    for c in cat_cols[:4]:
        print(f"  {c}:")
        print(df[c].value_counts(), "\n")

    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    fig.suptitle(f'EDA - {dataset_name}', fontsize=13, fontweight='bold')
    df[target_col].value_counts().plot(kind='bar', ax=axes[0],
        color=['#2ecc71','#e74c3c'], edgecolor='black')
    axes[0].set_title('Balance de Clases')
    axes[0].tick_params(axis='x', rotation=0)
    if num_cols:
        df[num_cols[0]].dropna().hist(bins=30, ax=axes[1], color='#3498db', edgecolor='black')
        axes[1].set_title(f'Distribucion: {num_cols[0]}')
    if len(num_cols) > 1:
        df.boxplot(column=num_cols[1], by=target_col, ax=axes[2])
        plt.sca(axes[2])
        plt.title(f'{num_cols[1]} vs Desercion')
    plt.tight_layout()
    plt.show()

    # -------------------------------------------------------
    # PASO 3: Mapa de correlacion
    # -------------------------------------------------------
    print("\n[PASO 3] MAPA DE CORRELACION")
    df_corr = df[num_cols].copy()
    df_corr['desercion_bin'] = (df[target_col] == 'Si').astype(int)
    corr_matrix = df_corr.corr()

    plt.figure(figsize=(min(14, len(df_corr.columns)+2), min(12, len(df_corr.columns)+2)))
    mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
    sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f',
                cmap='coolwarm', center=0, linewidths=.5, annot_kws={'size': 7})
    plt.title(f'Mapa de Correlacion - {dataset_name}')
    plt.tight_layout()
    plt.show()

    corr_target = corr_matrix['desercion_bin'].drop('desercion_bin').abs().sort_values(ascending=False)
    print("  Variables MAS correlacionadas con desercion:")
    print(corr_target.head(3).to_string())
    print("\n  Variables MENOS correlacionadas con desercion:")
    print(corr_target.tail(3).to_string())

    # -------------------------------------------------------
    # PASO 4: Preprocesamiento y pipelines
    # -------------------------------------------------------
    print("\n[PASO 4] PREPROCESAMIENTO - ColumnTransformer")
    X = df.drop(columns=[target_col])
    y = (df[target_col] == 'Si').astype(int)

    num_features = X.select_dtypes(include='number').columns.tolist()
    cat_features = X.select_dtypes(include='object').columns.tolist()
    print(f"  Numericas   ({len(num_features)}): {num_features[:5]}{'...' if len(num_features)>5 else ''}")
    print(f"  Categoricas ({len(cat_features)}): {cat_features[:5]}{'...' if len(cat_features)>5 else ''}")

    # Numeric: imputation(median) + StandardScaler
    numeric_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler',  StandardScaler())
    ])
    # Categorical: imputation(most_frequent) + OneHotEncoder
    categorical_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
    ])
    preprocessor = ColumnTransformer([
        ('num', numeric_pipeline,     num_features),
        ('cat', categorical_pipeline, cat_features)
    ])
    print("  Pipeline creado: Numeric(median+scaler) | Categorical(mode+OHE)")

    # -------------------------------------------------------
    # PASO 5: Train-test split
    # -------------------------------------------------------
    print("\n[PASO 5] TRAIN-TEST SPLIT 80/20")
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.20, random_state=42, stratify=y)
    print(f"  Train: {X_train.shape[0]} registros | Test: {X_test.shape[0]} registros")
    print(f"  Balance train -> No: {(y_train==0).sum()} | Si: {(y_train==1).sum()}")

    # -------------------------------------------------------
    # PASO 6: Modelos
    # -------------------------------------------------------
    print("\n[PASO 6] DEFINICION DE MODELOS")
    pipe_lr = Pipeline([
        ('prep',  preprocessor),
        ('model', LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42))
    ])
    pipe_dt = Pipeline([
        ('prep',  preprocessor),
        ('model', DecisionTreeClassifier(max_depth=6, class_weight='balanced', random_state=42))
    ])
    print("  Logistic Regression: max_iter=1000, class_weight=balanced")
    print("  Decision Tree      : max_depth=6,   class_weight=balanced")

    # -------------------------------------------------------
    # PASO 7: Validacion cruzada 5-fold
    # -------------------------------------------------------
    print("\n[PASO 7] VALIDACION CRUZADA 5-FOLD (scoring=f1)")
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    cv_lr = cross_val_score(pipe_lr, X_train, y_train, cv=cv, scoring='f1')
    cv_dt = cross_val_score(pipe_dt, X_train, y_train, cv=cv, scoring='f1')
    print(f"  Logistic Regression -> F1 por fold: {cv_lr.round(3)} | Media: {cv_lr.mean():.4f} +/- {cv_lr.std():.4f}")
    print(f"  Decision Tree       -> F1 por fold: {cv_dt.round(3)} | Media: {cv_dt.mean():.4f} +/- {cv_dt.std():.4f}")

    # -------------------------------------------------------
    # PASO 8: Metricas en test set
    # -------------------------------------------------------
    print("\n[PASO 8] METRICAS EN TEST SET")
    pipe_lr.fit(X_train, y_train)
    pipe_dt.fit(X_train, y_train)

    results = {}
    for name, pipe in [('Logistic Regression', pipe_lr), ('Decision Tree', pipe_dt)]:
        y_pred  = pipe.predict(X_test)
        y_proba = pipe.predict_proba(X_test)[:, 1]
        results[name] = {
            'y_pred': y_pred, 'y_proba': y_proba,
            'acc' : accuracy_score(y_test, y_pred),
            'prec': precision_score(y_test, y_pred, zero_division=0),
            'rec' : recall_score(y_test, y_pred, zero_division=0),
            'f1'  : f1_score(y_test, y_pred, zero_division=0),
            'auc' : roc_auc_score(y_test, y_proba)
        }
        print(f"\n  [{name}]")
        print(f"  Accuracy ={results[name]['acc']:.4f} | Precision={results[name]['prec']:.4f} | Recall={results[name]['rec']:.4f} | F1={results[name]['f1']:.4f} | AUC={results[name]['auc']:.4f}")
        print(classification_report(y_test, y_pred, target_names=['No deserta','Si deserta']))

    # Graficas
    fig, axes = plt.subplots(2, 3, figsize=(17, 10))
    fig.suptitle(f'Evaluacion de Modelos - {dataset_name}', fontsize=13, fontweight='bold')
    colors = {'Logistic Regression': '#3498db', 'Decision Tree': '#e74c3c'}

    for i, (name, res) in enumerate(results.items()):
        # Confusion matrix
        cm = confusion_matrix(y_test, res['y_pred'])
        ConfusionMatrixDisplay(cm, display_labels=['No deserta','Si deserta']).plot(
            ax=axes[i,0], colorbar=False, cmap='Blues')
        axes[i,0].set_title(f'Conf. Matrix - {name}')

        # ROC curve
        fpr, tpr, _ = roc_curve(y_test, res['y_proba'])
        axes[i,1].plot(fpr, tpr, color=colors[name], lw=2, label=f'AUC={res["auc"]:.3f}')
        axes[i,1].plot([0,1],[0,1],'k--',lw=1)
        axes[i,1].set_xlabel('FPR'); axes[i,1].set_ylabel('TPR')
        axes[i,1].set_title(f'ROC Curve - {name}')
        axes[i,1].legend(loc='lower right')
        axes[i,1].grid(alpha=.3)

    # Comparacion de metricas
    metricas = ['acc','prec','rec','f1','auc']
    labels   = ['Accuracy','Precision','Recall','F1','AUC']
    x = np.arange(len(labels)); w = 0.35
    axes[0,2].bar(x-w/2, [results['Logistic Regression'][m] for m in metricas], w,
                  label='Logistic Reg.', color='#3498db', edgecolor='black')
    axes[0,2].bar(x+w/2, [results['Decision Tree'][m] for m in metricas], w,
                  label='Decision Tree', color='#e74c3c', edgecolor='black')
    axes[0,2].set_xticks(x); axes[0,2].set_xticklabels(labels)
    axes[0,2].set_ylim(0, 1.1)
    axes[0,2].set_title('Comparacion de Metricas')
    axes[0,2].legend()
    axes[0,2].grid(axis='y', alpha=.3)

    # CV boxplot
    axes[1,2].boxplot([cv_lr, cv_dt], labels=['Logistic Reg.','Decision Tree'],
                      patch_artist=True, boxprops=dict(facecolor='lightblue'))
    axes[1,2].set_title('F1 Cross-Validation (5-fold)')
    axes[1,2].set_ylabel('F1-Score')
    axes[1,2].grid(axis='y', alpha=.3)
    plt.tight_layout()
    plt.show()

    # -------------------------------------------------------
    # PASO 9: Seleccion del modelo
    # -------------------------------------------------------
    print("\n[PASO 9] SELECCION DEL MEJOR MODELO")
    mejor = max(results, key=lambda k: results[k]['f1'])
    print(f"  Modelo seleccionado: {mejor}")
    print(f"  F1 Logistic Regression : {results['Logistic Regression']['f1']:.4f}")
    print(f"  F1 Decision Tree       : {results['Decision Tree']['f1']:.4f}")

    # Visualizacion del arbol de decision (primeros 3 niveles)
    try:
        fn_cat = (pipe_dt.named_steps['prep']
                  .transformers_[1][1]
                  .named_steps['encoder']
                  .get_feature_names_out(cat_features).tolist())
        feature_names = num_features + fn_cat
    except Exception:
        feature_names = None

    fig, ax = plt.subplots(figsize=(18, 6))
    plot_tree(pipe_dt.named_steps['model'], max_depth=3,
              feature_names=feature_names, class_names=['No','Si'],
              filled=True, rounded=True, fontsize=8, ax=ax)
    ax.set_title(f'Decision Tree (3 niveles) - {dataset_name}')
    plt.tight_layout()
    plt.show()

    # -------------------------------------------------------
    # PASO 10: Conclusiones
    # -------------------------------------------------------
    print("\n[PASO 10] CONCLUSIONES")
    lr = results['Logistic Regression']
    dt = results['Decision Tree']
    print(f"""
  Dataset analizado: {dataset_name}

  Logistic Regression
    Accuracy={lr['acc']:.3f} | Precision={lr['prec']:.3f} | Recall={lr['rec']:.3f} | F1={lr['f1']:.3f} | AUC={lr['auc']:.3f}
    Ventaja   : modelo simple, rapido e interpretable mediante sus coeficientes.
    Limitacion: asume linealidad entre las variables predictoras y el log-odds.

  Decision Tree
    Accuracy={dt['acc']:.3f} | Precision={dt['prec']:.3f} | Recall={dt['rec']:.3f} | F1={dt['f1']:.3f} | AUC={dt['auc']:.3f}
    Ventaja   : captura relaciones no lineales y es visualizable como reglas de decision.
    Limitacion: puede sobreajustar si no se controla la profundidad maxima.

  Modelo recomendado: {mejor}
    Se selecciona por mayor F1-Score, metrica que equilibra Precision y Recall.
    Esto es clave en un sistema de alerta temprana donde tanto las falsas alarmas
    como los casos de desercion no detectados tienen un costo real para la institucion.
    """)
    return results


## Ejecucion - Dataset Sintetico (70% calificacion)

In [ ]:
# CELDA 5 - Ejecutar los 10 pasos sobre el dataset sintetico
res_syn = run_pipeline(df_syn, dataset_name="Sintetico (Actividad I)")


## Ejecucion - Dataset Real del Profesor (30% calificacion)

In [ ]:
# CELDA 6 - Ejecutar los 10 pasos sobre el dataset real
# Asegurate de haber subido 'student_dropout__1_.csv' antes de correr esta celda
if df_real is not None:
    res_real = run_pipeline(df_real, dataset_name="Real (Profesor)")
else:
    print("Dataset real no disponible.")
    print("Sube 'student_dropout__1_.csv' y vuelve a ejecutar la Celda 3.")
